# PCF-VAE with SELFIES and Multi-Property Conditioning on ZINC-250k

This notebook trains:
- A SELFIES-based Masked Language Model (MLM) on SMILES from a ZINC-250k-style CSV with columns: `smiles, LogP, qed, SAS`.
- A PCF-VAE (posterior-collapse-resistant VAE with KL-annealing) conditioned on 3 properties: LogP, QED, SAS.
- A simple scaffold module that learns embeddings over Bemis–Murcko scaffolds (junction-tree-inspired layer).

It logs metrics per epoch with tqdm progress bars and plots curves at the end.

You can run this on a Kaggle T4 GPU in under ~12 hours by tuning `EPOCHS_MLM`, `EPOCHS_VAE`, `EPOCHS_SCAFFOLD`, and model sizes.

**Expected input CSV format**:
```text
smiles,LogP,qed,SAS
CCO,0.5,0.7,2.1
...
```

In [ ]:
%%capture
# Install required packages (Kaggle compatible)
!pip install selfies rdkit-pypi
!pip install tqdm

import os
import math
import json
import random
from typing import List, Tuple

import numpy as np
import pandas as pd
import selfies as sf
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Configuration and Paths

Edit `CSV_PATH` to point to your Kaggle input file. For example:

```python
CSV_PATH = "/kaggle/input/zinc250k-logp-qed-sas/zinc250k_logp_qed_sas.csv"
```

In [ ]:
# === User configuration ===
CSV_PATH = "/kaggle/input/zinc250k-logp-qed-sas/zinc250k_logp_qed_sas.csv"  # EDIT THIS
OUTPUT_DIR = "./models_zinc250k"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Data / tokenization
MAX_LEN = 100  # max SELFIES tokens
MIN_LEN = 5
VALID_FRACTION = 1.0  # use all valid rows

# Model sizes (keep moderate for 12h on T4)
EMBED_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS_MLM = 4
NUM_HEADS = 4

LATENT_DIM = 64
PROPERTY_DIM = 3  # LogP, QED, SAS

# Training hyperparameters
BATCH_SIZE = 256
LR_MLM = 1e-4
LR_VAE = 1e-4
LR_SCAFFOLD = 1e-4

EPOCHS_MLM = 10
EPOCHS_VAE = 20
EPOCHS_SCAFFOLD = 10

MASK_PROB = 0.15  # MLM mask probability

# KL annealing
BETA_START = 0.0
BETA_END = 1.0
BETA_WARMUP_EPOCHS = max(5, EPOCHS_VAE // 3)

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Config loaded.")

## Data Loading and SELFIES Conversion

- Load CSV with columns `[smiles, LogP, qed, SAS]`.
- Convert SMILES to SELFIES.
- Filter invalid or too-long sequences.
- Compute basic stats for properties.

In [ ]:
df = pd.read_csv(CSV_PATH)
assert set(["smiles", "LogP", "qed", "SAS"]).issubset(df.columns), "CSV must have columns: smiles, LogP, qed, SAS"
print("Loaded dataset shape:", df.shape)

def smiles_to_selfies_safe(smiles: str) -> str:
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        selfies = sf.encoder(smiles)
        return selfies
    except Exception:
        return None

selfies_list = []
valid_rows = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Converting SMILES to SELFIES"):
    s = str(row["smiles"]).strip()
    selfies = smiles_to_selfies_safe(s)
    if selfies is None:
        continue
    length = sf.len_selfies(selfies)
    if length < MIN_LEN or length > MAX_LEN:
        continue
    selfies_list.append(selfies)
    valid_rows.append(idx)

df_valid = df.loc[valid_rows].reset_index(drop=True)
df_valid["selfies"] = selfies_list
print("Valid filtered dataset:", df_valid.shape)

# Optionally subsample if needed
if VALID_FRACTION < 1.0:
    df_valid = df_valid.sample(frac=VALID_FRACTION, random_state=RANDOM_SEED).reset_index(drop=True)
    print("Subsampled dataset:", df_valid.shape)

prop_stats = df_valid[["LogP", "qed", "SAS"]].describe()
print(prop_stats)

df_valid.head()

## SELFIES Tokenizer

- Build vocabulary from SELFIES symbols.
- Include special tokens: `[PAD]`, `[SOS]`, `[EOS]`, `[MASK]`.
- Tokenizer converts SELFIES strings to padded sequences of token indices and back.

In [ ]:
from collections import Counter

class SelfiesTokenizer:
    def __init__(self, selfies_list: List[str], max_len: int = 100):
        self.max_len = max_len
        symbols = []
        for s in selfies_list:
            symbols.extend(list(sf.split_selfies(s)))
        counter = Counter(symbols)
        vocab_symbols = sorted(counter.keys())

        self.pad_token = "[PAD]"
        self.sos_token = "[SOS]"
        self.eos_token = "[EOS]"
        self.mask_token = "[MASK]"

        self.special_tokens = [self.pad_token, self.sos_token, self.eos_token, self.mask_token]
        self.itos = self.special_tokens + vocab_symbols
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    @property
    def vocab_size(self):
        return len(self.itos)

    def encode(self, selfies: str) -> List[int]:
        symbols = list(sf.split_selfies(selfies))
        tokens = [self.sos_token] + symbols + [self.eos_token]
        ids = [self.stoi.get(tok, self.stoi[self.mask_token]) for tok in tokens]
        if len(ids) < self.max_len:
            ids = ids + [self.stoi[self.pad_token]] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]
            if ids[-1] != self.stoi[self.eos_token]:
                ids[-1] = self.stoi[self.eos_token]
        return ids

    def decode(self, ids: List[int]) -> str:
        tokens = [self.itos[i] for i in ids]
        # remove PAD
        tokens = [t for t in tokens if t != self.pad_token]
        # strip SOS/EOS
        if self.sos_token in tokens:
            tokens = tokens[tokens.index(self.sos_token)+1:]
        if self.eos_token in tokens:
            tokens = tokens[:tokens.index(self.eos_token)]
        selfies = "".join(tokens)
        return selfies

tokenizer = SelfiesTokenizer(df_valid["selfies"].tolist(), max_len=MAX_LEN)
print("Vocab size:", tokenizer.vocab_size)

# Save vocab
vocab_path = os.path.join(OUTPUT_DIR, "selfies_vocab.json")
with open(vocab_path, "w") as f:
    json.dump({"itos": tokenizer.itos}, f)
print("Saved vocab to", vocab_path)

## Dataset and DataLoaders

We use a unified dataset that returns:
- input_ids (SELFIES token ids)
- property vector (LogP, QED, SAS) normalized
- scaffold id for scaffold module

MLM uses masked inputs; VAE uses full inputs and properties; scaffold module uses scaffold IDs.

In [ ]:
# Normalize properties (z-score)
prop_means = df_valid[["LogP", "qed", "SAS"]].mean().values.astype(np.float32)
prop_stds = df_valid[["LogP", "qed", "SAS"]].std().values.astype(np.float32) + 1e-6

def normalize_props(arr: np.ndarray) -> np.ndarray:
    return (arr - prop_means) / prop_stds

# Scaffold extraction
def get_scaffold(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "[NOSCAF]"
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    return scaffold if scaffold else "[NOSCAF]"

scaffolds = [get_scaffold(s) for s in df_valid["smiles"].tolist()]
unique_scaffolds = sorted(list(set(scaffolds)))
scaf2id = {s: i for i, s in enumerate(unique_scaffolds)}
id2scaf = {i: s for s, i in scaf2id.items()}
scaffold_ids = [scaf2id[s] for s in scaffolds]

print("Number of unique scaffolds:", len(unique_scaffolds))

class MoleculeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer: SelfiesTokenizer, scaffold_ids: List[int]):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.scaffold_ids = scaffold_ids

        self.props = self.df[["LogP", "qed", "SAS"]].values.astype(np.float32)
        self.props_norm = normalize_props(self.props)
        self.selfies = self.df["selfies"].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        selfies = self.selfies[idx]
        token_ids = self.tokenizer.encode(selfies)
        prop = self.props_norm[idx]
        scaf_id = self.scaffold_ids[idx]
        return {
            "input_ids": torch.tensor(token_ids, dtype=torch.long),
            "properties": torch.tensor(prop, dtype=torch.float),
            "scaffold_id": torch.tensor(scaf_id, dtype=torch.long)
        }

dataset = MoleculeDataset(df_valid, tokenizer, scaffold_ids)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print("Batches per epoch:", len(train_loader))

## MLM Model (SELFIES Masked Language Model)

A simple Transformer Encoder with token + positional embeddings.
We use a standard masked language modeling loss (cross-entropy over masked positions).

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

class MLMTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attention_mask=None):
        # input_ids: (batch, seq_len)
        x = self.token_emb(input_ids)
        x = self.pos_enc(x)
        if attention_mask is not None:
            # attention_mask: (batch, seq_len) -> (batch, seq_len) bool with True for keep
            key_padding_mask = ~attention_mask.bool()
        else:
            key_padding_mask = None
        h = self.encoder(x, src_key_padding_mask=key_padding_mask)
        logits = self.lm_head(h)
        return logits

mlm_model = MLMTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=EMBED_DIM,
    nhead=NUM_HEADS,
    num_layers=NUM_LAYERS_MLM,
    max_len=MAX_LEN
).to(device)

mlm_optimizer = torch.optim.Adam(mlm_model.parameters(), lr=LR_MLM)
mlm_history = {"loss": []}

pad_id = tokenizer.stoi[tokenizer.pad_token]
mask_id = tokenizer.stoi[tokenizer.mask_token]
sos_id = tokenizer.stoi[tokenizer.sos_token]
eos_id = tokenizer.stoi[tokenizer.eos_token]

def mask_inputs(input_ids: torch.Tensor, mask_prob: float) -> Tuple[torch.Tensor, torch.Tensor]:
    # input_ids: (batch, seq_len)
    labels = input_ids.clone()
    # do not mask PAD
    probability_matrix = torch.full(input_ids.shape, mask_prob, device=input_ids.device)
    special_mask = (input_ids == pad_id) | (input_ids == sos_id) | (input_ids == eos_id)
    probability_matrix.masked_fill_(special_mask, 0.0)
    masked_indices = torch.bernoulli(probability_matrix).bool()
    input_ids_masked = input_ids.clone()
    input_ids_masked[masked_indices] = mask_id
    # labels for non-masked positions = -100 so they are ignored by CE
    labels[~masked_indices] = -100
    return input_ids_masked, labels

def train_mlm_one_epoch(model, dataloader, optimizer):
    model.train()
    total_loss = 0.0
    n_tokens = 0
    loop = tqdm(dataloader, desc="MLM training")
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attn_mask = (input_ids != pad_id).to(device)
        masked_inputs, labels = mask_inputs(input_ids, MASK_PROB)
        logits = model(masked_inputs, attention_mask=attn_mask)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
            ignore_index=-100
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        batch_tokens = (labels != -100).sum().item()
        total_loss += loss.item() * batch_tokens
        n_tokens += batch_tokens
        loop.set_postfix({"loss": loss.item()})
    avg_loss = total_loss / max(n_tokens, 1)
    return avg_loss

print("MLM model ready.")

### Train MLM

We train for `EPOCHS_MLM` and save the best checkpoint based on lowest loss.

In [ ]:
best_mlm_loss = float("inf")
mlm_ckpt_path = os.path.join(OUTPUT_DIR, "mlm_selfies_best.pt")

for epoch in range(1, EPOCHS_MLM + 1):
    avg_loss = train_mlm_one_epoch(mlm_model, train_loader, mlm_optimizer)
    mlm_history["loss"].append(avg_loss)
    print(f"Epoch {epoch}/{EPOCHS_MLM} - MLM loss: {avg_loss:.4f}")
    if avg_loss < best_mlm_loss:
        best_mlm_loss = avg_loss
        torch.save({
            "model_state_dict": mlm_model.state_dict(),
            "config": {
                "vocab_size": tokenizer.vocab_size,
                "d_model": EMBED_DIM,
                "nhead": NUM_HEADS,
                "num_layers": NUM_LAYERS_MLM,
                "max_len": MAX_LEN
            },
            "history": mlm_history
        }, mlm_ckpt_path)
        print("Saved best MLM checkpoint.")

with open(os.path.join(OUTPUT_DIR, "mlm_history.json"), "w") as f:
    json.dump(mlm_history, f)
print("MLM training complete.")

## PCF-VAE (SELFIES Conditional VAE with KL-annealing)

- Encoder: uses MLM token embeddings (frozen or fine-tuned) + Transformer encoder.
- Latent: mean and log-variance vectors (dimension `LATENT_DIM`).
- Decoder: Transformer decoder conditioned on latent + properties.
- Loss: reconstruction (cross-entropy) + β(t) * KL divergence, with KL annealing to mitigate posterior collapse.

We treat this as a PCF-VAE-style model by combining KL-annealing and a stable architecture; you can add extra PCF regularizers if desired.

In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len, latent_dim, property_dim, mlm_token_emb=None):
        super().__init__()
        if mlm_token_emb is not None:
            self.token_emb = mlm_token_emb  # share embeddings
        else:
            self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.property_proj = nn.Linear(property_dim, d_model)
        self.fc_mu = nn.Linear(d_model, latent_dim)
        self.fc_logvar = nn.Linear(d_model, latent_dim)

    def forward(self, input_ids, properties, attention_mask=None):
        # input_ids: (batch, seq_len)
        x = self.token_emb(input_ids)
        x = self.pos_enc(x)
        if attention_mask is not None:
            key_padding_mask = ~attention_mask.bool()
        else:
            key_padding_mask = None
        h = self.encoder(x, src_key_padding_mask=key_padding_mask)  # (batch, seq_len, d_model)
        h_cls = h[:, 0, :]  # use first token as summary
        p = self.property_proj(properties)  # (batch, d_model)
        h_comb = h_cls + p
        mu = self.fc_mu(h_comb)
        logvar = self.fc_logvar(h_comb)
        return mu, logvar

class VAEDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len, latent_dim, property_dim, mlm_token_emb=None):
        super().__init__()
        if mlm_token_emb is not None:
            self.token_emb = mlm_token_emb
        else:
            self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        self.latent_proj = nn.Linear(latent_dim + property_dim, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, z, properties, attention_mask=None):
        # input_ids: (batch, seq_len) -> teacher-forced decoder input
        # z: (batch, latent_dim)
        # properties: (batch, property_dim)
        tgt = self.token_emb(input_ids)
        tgt = self.pos_enc(tgt)
        batch_size, seq_len, d_model = tgt.shape
        zp = torch.cat([z, properties], dim=-1)
        zp_proj = self.latent_proj(zp).unsqueeze(1).repeat(1, seq_len, 1)

        # causal mask for decoder
        tgt_mask = torch.triu(torch.ones(seq_len, seq_len, device=input_ids.device), diagonal=1).bool()

        h = self.decoder(tgt=tgt, memory=zp_proj, tgt_mask=tgt_mask)
        logits = self.lm_head(h)
        return logits

class PCFVAE(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers_enc, num_layers_dec, max_len, latent_dim, property_dim, mlm_model=None):
        super().__init__()
        mlm_token_emb = mlm_model.token_emb if mlm_model is not None else None
        self.encoder = VAEEncoder(vocab_size, d_model, nhead, num_layers_enc, max_len, latent_dim, property_dim, mlm_token_emb=mlm_token_emb)
        self.decoder = VAEDecoder(vocab_size, d_model, nhead, num_layers_dec, max_len, latent_dim, property_dim, mlm_token_emb=mlm_token_emb)

    def encode(self, input_ids, properties, attention_mask=None):
        mu, logvar = self.encoder(input_ids, properties, attention_mask)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, input_ids, z, properties, attention_mask=None):
        logits = self.decoder(input_ids, z, properties, attention_mask)
        return logits

    def forward(self, input_ids, properties, attention_mask=None):
        mu, logvar = self.encode(input_ids, properties, attention_mask)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(input_ids, z, properties, attention_mask)
        return logits, mu, logvar

vae_model = PCFVAE(
    vocab_size=tokenizer.vocab_size,
    d_model=EMBED_DIM,
    nhead=NUM_HEADS,
    num_layers_enc=2,
    num_layers_dec=2,
    max_len=MAX_LEN,
    latent_dim=LATENT_DIM,
    property_dim=PROPERTY_DIM,
    mlm_model=mlm_model
).to(device)

vae_optimizer = torch.optim.Adam(vae_model.parameters(), lr=LR_VAE)
vae_history = {"recon_loss": [], "kl_loss": [], "beta": []}

def kl_annealing(epoch):
    if epoch <= BETA_WARMUP_EPOCHS:
        return BETA_START + (BETA_END - BETA_START) * (epoch / BETA_WARMUP_EPOCHS)
    else:
        return BETA_END

def vae_loss_fn(logits, input_ids, mu, logvar, beta):
    # reconstruction: cross-entropy over all tokens (except PAD)
    recon_targets = input_ids.view(-1)
    logits_flat = logits.view(-1, logits.size(-1))
    recon_loss = F.cross_entropy(logits_flat, recon_targets, ignore_index=pad_id)
    # KL divergence between q(z|x) and N(0,I)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss.detach(), kl_loss.detach()

def train_vae_one_epoch(model, dataloader, optimizer, epoch):
    model.train()
    beta = kl_annealing(epoch)
    total_recon = 0.0
    total_kl = 0.0
    n_batches = 0
    loop = tqdm(dataloader, desc=f"VAE training (epoch {epoch})")
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        properties = batch["properties"].to(device)
        attn_mask = (input_ids != pad_id).to(device)
        logits, mu, logvar = model(input_ids, properties, attn_mask)
        loss, recon_l, kl_l = vae_loss_fn(logits, input_ids, mu, logvar, beta)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_recon += recon_l.item()
        total_kl += kl_l.item()
        n_batches += 1
        loop.set_postfix({"loss": loss.item(), "recon": recon_l.item(), "kl": kl_l.item(), "beta": beta})
    avg_recon = total_recon / max(n_batches, 1)
    avg_kl = total_kl / max(n_batches, 1)
    return avg_recon, avg_kl, beta

print("PCF-VAE model ready.")

### Train PCF-VAE

We train for `EPOCHS_VAE` and save the best checkpoint based on lowest total loss (reconstruction + β*KL).

In [ ]:
best_vae_loss = float("inf")
vae_ckpt_path = os.path.join(OUTPUT_DIR, "pcf_vae_selfies_best.pt")

for epoch in range(1, EPOCHS_VAE + 1):
    avg_recon, avg_kl, beta = train_vae_one_epoch(vae_model, train_loader, vae_optimizer, epoch)
    vae_history["recon_loss"].append(avg_recon)
    vae_history["kl_loss"].append(avg_kl)
    vae_history["beta"].append(beta)
    total = avg_recon + beta * avg_kl
    print(f"Epoch {epoch}/{EPOCHS_VAE} - VAE recon: {avg_recon:.4f}, KL: {avg_kl:.4f}, beta: {beta:.3f}, total: {total:.4f}")
    if total < best_vae_loss:
        best_vae_loss = total
        torch.save({
            "model_state_dict": vae_model.state_dict(),
            "config": {
                "vocab_size": tokenizer.vocab_size,
                "d_model": EMBED_DIM,
                "nhead": NUM_HEADS,
                "num_layers_enc": 2,
                "num_layers_dec": 2,
                "max_len": MAX_LEN,
                "latent_dim": LATENT_DIM,
                "property_dim": PROPERTY_DIM
            },
            "history": vae_history,
            "prop_means": prop_means.tolist(),
            "prop_stds": prop_stds.tolist()
        }, vae_ckpt_path)
        print("Saved best PCF-VAE checkpoint.")

with open(os.path.join(OUTPUT_DIR, "vae_history.json"), "w") as f:
    json.dump(vae_history, f)
print("PCF-VAE training complete.")

## Scaffold (Junction-Tree–Inspired) Module

We train a simple scaffold autoencoder:
- Input: scaffold ID (categorical) and normalized properties.
- Encoder: learns a latent scaffold embedding.
- Decoder: reconstructs scaffold ID (classification).

This is a lightweight, junction-tree-inspired component you can later extend to a full JT-VAE if needed.

In [ ]:
NUM_SCAFFOLDS = len(unique_scaffolds)

class ScaffoldAutoEncoder(nn.Module):
    def __init__(self, num_scaffolds, property_dim, embed_dim=128, latent_dim=64):
        super().__init__()
        self.scaf_emb = nn.Embedding(num_scaffolds, embed_dim)
        self.encoder = nn.Sequential(
            nn.Linear(embed_dim + property_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, num_scaffolds)
        )

    def forward(self, scaf_ids, properties):
        # scaf_ids: (batch,), properties: (batch, property_dim)
        emb = self.scaf_emb(scaf_ids)
        x = torch.cat([emb, properties], dim=-1)
        z = self.encoder(x)
        logits = self.decoder(z)
        return logits, z

scaffold_model = ScaffoldAutoEncoder(NUM_SCAFFOLDS, PROPERTY_DIM).to(device)
scaffold_optimizer = torch.optim.Adam(scaffold_model.parameters(), lr=LR_SCAFFOLD)
scaffold_history = {"loss": []}

def train_scaffold_one_epoch(model, dataloader, optimizer):
    model.train()
    total_loss = 0.0
    n_batches = 0
    loop = tqdm(dataloader, desc="Scaffold training")
    for batch in loop:
        scaf_ids = batch["scaffold_id"].to(device)
        properties = batch["properties"].to(device)
        logits, _ = model(scaf_ids, properties)
        loss = F.cross_entropy(logits, scaf_ids)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
        loop.set_postfix({"loss": loss.item()})
    avg_loss = total_loss / max(n_batches, 1)
    return avg_loss

print("Scaffold model ready.")

### Train Scaffold Module

Train for `EPOCHS_SCAFFOLD` and save best checkpoint based on lowest loss.

In [ ]:
best_scaffold_loss = float("inf")
scaffold_ckpt_path = os.path.join(OUTPUT_DIR, "scaffold_autoencoder_best.pt")

for epoch in range(1, EPOCHS_SCAFFOLD + 1):
    avg_loss = train_scaffold_one_epoch(scaffold_model, train_loader, scaffold_optimizer)
    scaffold_history["loss"].append(avg_loss)
    print(f"Epoch {epoch}/{EPOCHS_SCAFFOLD} - Scaffold loss: {avg_loss:.4f}")
    if avg_loss < best_scaffold_loss:
        best_scaffold_loss = avg_loss
        torch.save({
            "model_state_dict": scaffold_model.state_dict(),
            "config": {
                "num_scaffolds": NUM_SCAFFOLDS,
                "property_dim": PROPERTY_DIM
            },
            "history": scaffold_history
        }, scaffold_ckpt_path)
        print("Saved best scaffold checkpoint.")

with open(os.path.join(OUTPUT_DIR, "scaffold_history.json"), "w") as f:
    json.dump(scaffold_history, f)
print("Scaffold training complete.")

## Simple Generation and Validity Check

We generate molecules from the trained PCF-VAE for random property targets and compute a simple RDKit validity metric for a small sample (for sanity check).

In [ ]:
def sample_properties(n_samples: int = 32):
    # sample normalized properties from N(0,1)
    return torch.randn(n_samples, PROPERTY_DIM, device=device)

def generate_selfies(vae_model, tokenizer, properties, max_len=MAX_LEN):
    vae_model.eval()
    with torch.no_grad():
        batch_size = properties.size(0)
        # start from SOS
        input_ids = torch.full((batch_size, 1), sos_id, dtype=torch.long, device=device)
        mu, logvar = vae_model.encoder(input_ids, properties)
        z = vae_model.reparameterize(mu, logvar)
        generated = [ [sos_id] for _ in range(batch_size) ]
        for t in range(1, max_len):
            dec_input = torch.tensor(generated, dtype=torch.long, device=device)
            logits = vae_model.decode(dec_input, z, properties)
            next_token = torch.argmax(logits[:, -1, :], dim=-1)
            for i in range(batch_size):
                generated[i].append(next_token[i].item())
        selfies_list = []
        for ids in generated:
            selfies = tokenizer.decode(ids)
            selfies_list.append(selfies)
        return selfies_list

def selfies_to_smiles_safe(selfies: str) -> str:
    try:
        smi = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol)
    except Exception:
        return None

n_samples = 64
props = sample_properties(n_samples)
gen_selfies = generate_selfies(vae_model, tokenizer, props)
gen_smiles = []
valid_count = 0
for s in gen_selfies:
    smi = selfies_to_smiles_safe(s)
    if smi is not None:
        valid_count += 1
        gen_smiles.append(smi)

print(f"Generated {n_samples} molecules, RDKit-valid: {valid_count} ({valid_count/n_samples*100:.1f}%)")
print("Example generated SMILES:", gen_smiles[:10])

## Plotting Metrics for the Paper

We plot:
- MLM training loss vs epoch.
- VAE reconstruction loss, KL loss, and β vs epoch.
- Scaffold reconstruction loss vs epoch.

You can export these figures as PNGs for your paper.

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(mlm_history["loss"], marker='o')
plt.xlabel("Epoch")
plt.ylabel("MLM loss")
plt.title("MLM training loss")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mlm_loss.png"), dpi=300)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(vae_history["recon_loss"], label="Recon loss", marker='o')
plt.plot(vae_history["kl_loss"], label="KL loss", marker='s')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("PCF-VAE losses")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "vae_losses.png"), dpi=300)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(vae_history["beta"], label="beta", marker='^')
plt.xlabel("Epoch")
plt.ylabel("Beta")
plt.title("KL annealing schedule")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "beta_schedule.png"), dpi=300)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(scaffold_history["loss"], marker='o')
plt.xlabel("Epoch")
plt.ylabel("Scaffold recon loss")
plt.title("Scaffold autoencoder loss")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "scaffold_loss.png"), dpi=300)
plt.show()

print("All training complete. Models and plots saved in:", OUTPUT_DIR)